# Gene-Level Tables EDA
**DNA Gene Mapping Project - ML Phase V5**  
**Author:** Sharique Mohammad  
**Date:** February 2026  
**Tables covered:**
- gene_expression_ml_features (44,874 genes, 51 cols)
- gene_pharmacogene_ml_features (2,209 genes, 66 cols)
- gene_protein_family_ml_features (44,874 genes, 57 cols)
- gene_test_availability_ml_features (44,874 genes, 47 cols)
- genetic_test_ml_features (44,874 genes, 25 cols)
- protein_family_ml_features (44,874 genes, 34 cols)
- transcript_expression_ml_features (44,874 genes, 26 cols)

## Objective
EDA on all gene-level tables. These are small datasets (2K to 44K rows) requiring cross-validation for model training. Focus on class balance, feature distributions, and cross-validation suitability.

## Use Cases Covered
- UC10: Gene Pharmacogene Priority (target: is_high_priority_pharmacogene, ~2,209 rows - use CV)
- UC11: Gene Expression Relevance (target: is_clinically_relevant_expression)
- UC12: Protein Family Druggability (target: is_high_value_protein_family)
- UC13: Genetic Test Availability Priority (target: is_high_priority_test_gene)
- UC14: Expression Pattern (target: is_clinically_relevant_expression)

## Key Note
gene_pharmacogene_ml_features has only ~2,209 rows. Cross-validation required instead of hold-out splits.

## Deliverables
- Visualizations and metrics saved per table under respective analytical folders

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
from pathlib import Path
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

PROJECT_ROOT = Path().absolute().parent.parent
ANALYTICAL   = PROJECT_ROOT / 'data' / 'analytical'

def make_dirs(table_name):
    base = ANALYTICAL / table_name
    (base / 'images').mkdir(parents=True, exist_ok=True)
    (base / 'reports').mkdir(parents=True, exist_ok=True)
    (base / 'metrics').mkdir(parents=True, exist_ok=True)
    return base / 'images', base / 'reports', base / 'metrics'

def save_metrics(df, corr_feats, metrics_dir):
    miss = pd.DataFrame({'column': df.columns,
                          'missing_pct': (df.isnull().sum().values / len(df) * 100).round(2)
                         }).sort_values('missing_pct', ascending=False)
    miss.to_csv(metrics_dir / 'missing_values.csv', index=False)
    cf = [c for c in corr_feats if c in df.columns]
    df[cf].apply(pd.to_numeric, errors='coerce').corr().to_csv(metrics_dir / 'correlation_matrix.csv')
    df[cf].describe().T.to_csv(metrics_dir / 'feature_statistics.csv')
    return miss

print("Setup complete")

## 2. Database Connection

In [ ]:
POSTGRES_HOST     = os.getenv("POSTGRES_HOST")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT")
POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)

print("Database connection established")
print(f"Host : {POSTGRES_HOST}:{POSTGRES_PORT}")
print(f"DB   : {POSTGRES_DB}")

---
## 3. gene_expression_ml_features
**Use Case 11 — Gene Expression Relevance**
**Target:** is_clinically_relevant_expression  (44,874 genes)

In [ ]:
IMAGES_DIR, REPORTS_DIR, METRICS_DIR = make_dirs('gene_expression_ml_features')

print("Loading gene_expression_ml_features (full table)...")
df_ge = pd.read_sql("SELECT * FROM gold.gene_expression_ml_features", engine)
print(f"Rows: {len(df_ge):,}  Cols: {len(df_ge.columns)}")

int_cols_ge = [
    'gene_length', 'total_tissues_expressed', 'tissue_type_count',
    'primary_tissue_count', 'expression_significance_score', 'clinical_relevance_score',
    'total_disease_count', 'disease_category_count', 'cancer_mutation_count',
    'unique_tumor_samples', 'max_domain_count', 'has_kinase_domain_count',
    'total_gene_variants', 'splice_variants', 'expression_affecting_variants',
    'disease_expression_score', 'cancer_expression_score', 'functional_expression_score'
]
double_cols_ge = ['druggability_score', 'max_expression_tpm', 'avg_expression_tpm',
                   'peak_expression_tpm', 'tissue_specificity_score']
bool_cols_ge = [
    'is_kinase', 'is_receptor', 'is_enzyme', 'is_transcription_factor',
    'is_pharmacogene', 'is_ubiquitously_expressed', 'is_tissue_specific',
    'is_highly_expressed', 'is_lowly_expressed', 'has_cancer_disease',
    'has_neurological_disease', 'has_metabolic_disease', 'is_disease_gene',
    'is_cancer_gene', 'has_functional_domain', 'has_expression_variants',
    'is_clinically_relevant_expression'
]

for col in int_cols_ge:
    if col in df_ge.columns:
        df_ge[col] = pd.to_numeric(df_ge[col], errors='coerce').astype('Int64')
for col in double_cols_ge:
    if col in df_ge.columns:
        df_ge[col] = pd.to_numeric(df_ge[col], errors='coerce')
for col in bool_cols_ge:
    if col in df_ge.columns:
        df_ge[col] = df_ge[col].astype(str).str.lower().map({'true': True, 'false': False})

total_ge = len(df_ge)
target_ge = int(df_ge['is_clinically_relevant_expression'].sum()) if 'is_clinically_relevant_expression' in df_ge.columns else 0
print(f"\nTarget is_clinically_relevant_expression : {target_ge:,} ({target_ge/total_ge*100:.1f}%)")
if target_ge > 0 and (total_ge - target_ge) > 0:
    ratio_ge = max(target_ge, total_ge-target_ge) / min(target_ge, total_ge-target_ge)
    print(f"Imbalance : {ratio_ge:.2f}:1  SMOTE: {ratio_ge > 5}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].bar(['Clinically Relevant', 'Not Relevant'],
            [target_ge, total_ge - target_ge],
            color=['#e74c3c', '#27ae60'], alpha=0.8, edgecolor='black')
for i, val in enumerate([target_ge, total_ge - target_ge]):
    axes[0].text(i, val, f'{val:,}\n({val/total_ge*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('UC11: is_clinically_relevant_expression', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

if 'expression_breadth_category' in df_ge.columns:
    eb_dist = df_ge['expression_breadth_category'].value_counts()
    eb_dist.sort_values().plot(kind='barh', ax=axes[1], color='teal', alpha=0.8, edgecolor='black')
    axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[1].set_title('Expression Breadth Category', fontsize=12, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '01_target_expression_breadth.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_target_expression_breadth.png")

score_feats_ge = ['max_expression_tpm', 'avg_expression_tpm', 'tissue_specificity_score',
                   'druggability_score', 'clinical_relevance_score', 'disease_expression_score']
score_feats_ge = [c for c in score_feats_ge if c in df_ge.columns]
n_cols = 3
n_rows = (len(score_feats_ge) + 2) // 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()
for i, col in enumerate(score_feats_ge):
    data = df_ge[col].dropna()
    axes[i].hist(data, bins=40, color='steelblue', alpha=0.8, edgecolor='black')
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Value', fontsize=9)
    axes[i].set_ylabel('Frequency', fontsize=9)
    axes[i].grid(alpha=0.3)
for i in range(len(score_feats_ge), len(axes)):
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(IMAGES_DIR / '02_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 02_score_distributions.png")

save_metrics(df_ge, score_feats_ge + ['total_tissues_expressed','cancer_mutation_count','total_gene_variants'], METRICS_DIR)
print(f"Saved metrics to {METRICS_DIR}")

---
## 4. gene_pharmacogene_ml_features
**Use Case 10 — Gene Pharmacogene Priority**
**Target:** is_high_priority_pharmacogene  (2,209 genes - CROSS-VALIDATION REQUIRED)

In [ ]:
IMAGES_DIR, REPORTS_DIR, METRICS_DIR = make_dirs('gene_pharmacogene_ml_features')

print("Loading gene_pharmacogene_ml_features (full table - only 2,209 rows)...")
df_gpg = pd.read_sql("SELECT * FROM gold.gene_pharmacogene_ml_features", engine)
print(f"Rows: {len(df_gpg):,}  Cols: {len(df_gpg.columns)}")
print("NOTE: Cross-validation required due to small dataset size")

int_cols_gpg = [
    'source_count', 'total_relationships', 'entity_type_count', 'drug_relationships',
    'disease_relationships', 'variant_relationships', 'evidence_count',
    'total_gene_variants', 'pathogenic_variants', 'missense_variants',
    'lof_variants', 'domain_affecting_variants', 'tissues_expressed_count',
    'cancer_mutation_count', 'unique_tumor_samples', 'total_disease_count',
    'max_domain_count', 'has_kinase_domain_count', 'pharmacogene_evidence_score',
    'drug_interaction_score', 'pharmacogene_variant_impact_score',
    'metabolism_context_score'
]
double_cols_gpg = ['druggability_score', 'avg_pathogenicity_score',
                    'max_expression_tpm', 'avg_expression_tpm', 'clinical_utility_score']
bool_cols_gpg = [
    'has_pharmgkb_annotation', 'is_drug_metabolizer', 'is_drug_transporter_gene',
    'is_drug_target_gene', 'has_high_druggability', 'is_pharmacogene',
    'is_hepatic_metabolizer', 'is_renal_transporter', 'is_validated_cancer_target',
    'is_kinase', 'is_receptor', 'is_enzyme', 'is_transporter', 'is_metabolic',
    'has_pharmacogene_variants', 'is_liver_expressed', 'is_kidney_expressed',
    'is_oncology_drug_target', 'has_cancer_disease', 'has_cardiovascular_disease',
    'has_neurological_disease', 'has_metabolic_disease', 'is_complex_drug_target',
    'is_high_priority_pharmacogene'
]

for col in int_cols_gpg:
    if col in df_gpg.columns:
        df_gpg[col] = pd.to_numeric(df_gpg[col], errors='coerce').astype('Int64')
for col in double_cols_gpg:
    if col in df_gpg.columns:
        df_gpg[col] = pd.to_numeric(df_gpg[col], errors='coerce')
for col in bool_cols_gpg:
    if col in df_gpg.columns:
        df_gpg[col] = df_gpg[col].astype(str).str.lower().map({'true': True, 'false': False})

total_gpg = len(df_gpg)
target_gpg = int(df_gpg['is_high_priority_pharmacogene'].sum()) if 'is_high_priority_pharmacogene' in df_gpg.columns else 0
print(f"\nTarget is_high_priority_pharmacogene : {target_gpg:,} ({target_gpg/total_gpg*100:.1f}%)")
print(f"Train split (~70%)                   : {int(total_gpg * 0.7):,} rows")
print(f"Recommendation                       : Use 5-fold cross-validation")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].bar(['High Priority', 'Not High Priority'],
            [target_gpg, total_gpg - target_gpg],
            color=['#e74c3c', '#27ae60'], alpha=0.8, edgecolor='black')
for i, val in enumerate([target_gpg, total_gpg - target_gpg]):
    axes[0].text(i, val, f'{val:,}\n({val/total_gpg*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('UC10: is_high_priority_pharmacogene', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

if 'pharmacogene_category' in df_gpg.columns:
    pgcat = df_gpg['pharmacogene_category'].value_counts()
    pgcat.sort_values().plot(kind='barh', ax=axes[1], color='steelblue', alpha=0.8, edgecolor='black')
    axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[1].set_title('Pharmacogene Category', fontsize=12, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '01_target_pharmacogene_category.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_target_pharmacogene_category.png")

score_feats_gpg = ['druggability_score', 'avg_pathogenicity_score', 'max_expression_tpm',
                    'clinical_utility_score', 'pharmacogene_evidence_score',
                    'drug_interaction_score', 'pharmacogene_variant_impact_score']
score_feats_gpg = [c for c in score_feats_gpg if c in df_gpg.columns]

n_cols = 3
n_rows = (len(score_feats_gpg) + 2) // 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()
for i, col in enumerate(score_feats_gpg):
    data = df_gpg[col].dropna()
    axes[i].hist(data, bins=30, color='coral', alpha=0.8, edgecolor='black')
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Value', fontsize=9)
    axes[i].set_ylabel('Frequency', fontsize=9)
    axes[i].grid(alpha=0.3)
for i in range(len(score_feats_gpg), len(axes)):
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(IMAGES_DIR / '02_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 02_score_distributions.png")

save_metrics(df_gpg, score_feats_gpg + ['source_count','total_relationships','tissues_expressed_count'], METRICS_DIR)
print(f"Saved metrics to {METRICS_DIR}")

---
## 5. gene_protein_family_ml_features
**Use Case 12 — Protein Family Druggability**
**Target:** is_high_value_protein_family  (44,874 genes)

In [ ]:
IMAGES_DIR, REPORTS_DIR, METRICS_DIR = make_dirs('gene_protein_family_ml_features')

print("Loading gene_protein_family_ml_features (full table)...")
df_gpf = pd.read_sql("SELECT * FROM gold.gene_protein_family_ml_features", engine)
print(f"Rows: {len(df_gpf):,}  Cols: {len(df_gpf.columns)}")

int_cols_gpf = [
    'protein_count', 'max_domain_count', 'proteins_with_kinase', 'proteins_with_receptor',
    'proteins_with_zinc_finger', 'proteins_with_sh2', 'proteins_with_sh3',
    'proteins_with_ph', 'proteins_with_death', 'proteins_with_leucine_zipper',
    'proteins_with_helix_loop', 'proteins_with_ig', 'proteins_with_functional_domain',
    'domain_diversity_score', 'functional_complexity_score', 'druggability_potential_score',
    'domain_affecting_variants', 'domain_pathogenic_variants', 'critical_domain_variants',
    'protein_family_expression_breadth', 'cancer_missense_mutations',
    'cancer_truncating_mutations', 'cancer_samples_affected',
    'total_disease_count', 'variant_domain_impact_score'
]
double_cols_gpf = ['druggability_score', 'protein_max_expression',
                    'cancer_protein_family_score', 'disease_protein_family_score']
bool_cols_gpf = [
    'is_kinase', 'is_receptor', 'is_enzyme', 'is_pharmacogene',
    'has_signaling_domain', 'has_dna_binding_domain', 'has_membrane_domain',
    'has_apoptosis_domain', 'has_immune_domain', 'is_multi_domain_protein',
    'has_domain_variants', 'tissue_specific_protein_expression',
    'cancer_relevant_protein_family', 'has_cancer_disease',
    'has_neurological_disease', 'disease_associated_protein_family',
    'is_high_value_protein_family'
]

for col in int_cols_gpf:
    if col in df_gpf.columns:
        df_gpf[col] = pd.to_numeric(df_gpf[col], errors='coerce').astype('Int64')
for col in double_cols_gpf:
    if col in df_gpf.columns:
        df_gpf[col] = pd.to_numeric(df_gpf[col], errors='coerce')
for col in bool_cols_gpf:
    if col in df_gpf.columns:
        df_gpf[col] = df_gpf[col].astype(str).str.lower().map({'true': True, 'false': False})

total_gpf = len(df_gpf)
target_gpf = int(df_gpf['is_high_value_protein_family'].sum()) if 'is_high_value_protein_family' in df_gpf.columns else 0
print(f"\nTarget is_high_value_protein_family : {target_gpf:,} ({target_gpf/total_gpf*100:.1f}%)")
if target_gpf > 0 and (total_gpf - target_gpf) > 0:
    ratio_gpf = max(target_gpf, total_gpf-target_gpf) / min(target_gpf, total_gpf-target_gpf)
    print(f"Imbalance : {ratio_gpf:.2f}:1  SMOTE: {ratio_gpf > 5}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].bar(['High Value', 'Not High Value'], [target_gpf, total_gpf - target_gpf],
            color=['#e74c3c', '#27ae60'], alpha=0.8, edgecolor='black')
for i, val in enumerate([target_gpf, total_gpf - target_gpf]):
    axes[0].text(i, val, f'{val:,}\n({val/total_gpf*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('UC12: is_high_value_protein_family', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

if 'protein_family_priority' in df_gpf.columns:
    pfp_dist = df_gpf['protein_family_priority'].value_counts()
    pfp_dist.sort_values().plot(kind='barh', ax=axes[1], color='mediumpurple', alpha=0.8, edgecolor='black')
    axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[1].set_title('Protein Family Priority Distribution', fontsize=12, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '01_target_protein_family.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_target_protein_family.png")

dom_feats = ['domain_diversity_score','functional_complexity_score','druggability_potential_score',
             'druggability_score','cancer_protein_family_score','disease_protein_family_score']
dom_feats = [c for c in dom_feats if c in df_gpf.columns]
n_cols = 3
n_rows = (len(dom_feats) + 2) // 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()
for i, col in enumerate(dom_feats):
    data = df_gpf[col].dropna()
    axes[i].hist(data, bins=30, color='teal', alpha=0.8, edgecolor='black')
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Value', fontsize=9)
    axes[i].set_ylabel('Frequency', fontsize=9)
    axes[i].grid(alpha=0.3)
for i in range(len(dom_feats), len(axes)):
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(IMAGES_DIR / '02_domain_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 02_domain_scores.png")

save_metrics(df_gpf, dom_feats + ['protein_count','max_domain_count','domain_affecting_variants'], METRICS_DIR)
print(f"Saved metrics to {METRICS_DIR}")

---
## 6. gene_test_availability_ml_features and genetic_test_ml_features
**Use Case 13 — Genetic Test Availability Priority**
**Target:** is_high_priority_test_gene  (44,874 genes)

In [ ]:
IMAGES_DIR_GTA, REPORTS_DIR_GTA, METRICS_DIR_GTA = make_dirs('gene_test_availability_ml_features')
IMAGES_DIR_GT, REPORTS_DIR_GT, METRICS_DIR_GT   = make_dirs('genetic_test_ml_features')

print("Loading gene_test_availability_ml_features (full table)...")
df_gta = pd.read_sql("SELECT * FROM gold.gene_test_availability_ml_features", engine)
print(f"Rows: {len(df_gta):,}  Cols: {len(df_gta.columns)}")

int_cols_gta = [
    'total_test_count', 'unique_test_count', 'disease_count', 'genetic_test_count',
    'tests_with_gene_info', 'tests_with_disease_info', 'complete_test_count',
    'frequent_test_count', 'test_accessibility_score', 'clinical_utility_score',
    'test_quality_score', 'total_disease_count', 'pathogenic_variants_in_tested_gene',
    'test_covered_variants', 'cancer_mutation_count', 'cancer_samples',
    'rare_pathogenic_variants', 'clinical_test_utility_score',
    'variant_test_coverage_score', 'population_test_relevance_score'
]
bool_cols_gta = [
    'is_kinase', 'is_receptor', 'is_enzyme', 'is_pharmacogene',
    'has_clinical_test', 'has_multiple_tests', 'has_comprehensive_testing',
    'is_well_tested_gene', 'has_cancer_disease', 'has_cardiovascular_disease',
    'has_neurological_disease', 'multi_disease_testing', 'is_cancer_panel_gene',
    'hereditary_cancer_testing', 'carrier_screening_relevant', 'is_high_priority_test_gene'
]

for col in int_cols_gta:
    if col in df_gta.columns:
        df_gta[col] = pd.to_numeric(df_gta[col], errors='coerce').astype('Int64')
for col in bool_cols_gta:
    if col in df_gta.columns:
        df_gta[col] = df_gta[col].astype(str).str.lower().map({'true': True, 'false': False})

total_gta = len(df_gta)
target_gta = int(df_gta['is_high_priority_test_gene'].sum()) if 'is_high_priority_test_gene' in df_gta.columns else 0
print(f"\nTarget is_high_priority_test_gene : {target_gta:,} ({target_gta/total_gta*100:.1f}%)")
if target_gta > 0 and (total_gta - target_gta) > 0:
    ratio_gta = max(target_gta, total_gta-target_gta) / min(target_gta, total_gta-target_gta)
    print(f"Imbalance : {ratio_gta:.2f}:1  SMOTE: {ratio_gta > 5}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].bar(['High Priority', 'Not High Priority'], [target_gta, total_gta - target_gta],
            color=['#e74c3c', '#27ae60'], alpha=0.8, edgecolor='black')
for i, val in enumerate([target_gta, total_gta - target_gta]):
    axes[0].text(i, val, f'{val:,}\n({val/total_gta*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('UC13: is_high_priority_test_gene', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

if 'test_priority' in df_gta.columns:
    tp_dist = df_gta['test_priority'].value_counts()
    tp_dist.sort_values().plot(kind='barh', ax=axes[1], color='steelblue', alpha=0.8, edgecolor='black')
    axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[1].set_title('Test Priority Distribution', fontsize=12, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR_GTA / '01_target_test_priority.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: gene_test_availability_ml_features/01_target_test_priority.png")

test_feats = ['total_test_count','unique_test_count','test_accessibility_score',
              'clinical_utility_score','test_quality_score','clinical_test_utility_score']
test_feats = [c for c in test_feats if c in df_gta.columns]
n_cols = 3
n_rows = (len(test_feats) + 2) // 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()
for i, col in enumerate(test_feats):
    data = df_gta[col].dropna()
    axes[i].hist(data, bins=30, color='steelblue', alpha=0.8, edgecolor='black')
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Value', fontsize=9)
    axes[i].set_ylabel('Frequency', fontsize=9)
    axes[i].grid(alpha=0.3)
for i in range(len(test_feats), len(axes)):
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(IMAGES_DIR_GTA / '02_test_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: gene_test_availability_ml_features/02_test_score_distributions.png")

save_metrics(df_gta, test_feats, METRICS_DIR_GTA)

# genetic_test_ml_features (simpler version)
print("\nLoading genetic_test_ml_features (full table)...")
df_gt = pd.read_sql("SELECT * FROM gold.genetic_test_ml_features", engine)
print(f"Rows: {len(df_gt):,}  Cols: {len(df_gt.columns)}")

int_cols_gt = ['total_test_count','unique_test_count','disease_count','genetic_test_count',
               'tests_with_gene_info','tests_with_disease_info','complete_test_count',
               'frequent_test_count','test_accessibility_score','clinical_utility_score','test_quality_score']
bool_cols_gt = ['is_kinase','is_receptor','is_enzyme','has_clinical_test',
                'has_multiple_tests','has_comprehensive_testing','is_well_tested_gene','is_high_priority_test_gene']
for col in int_cols_gt:
    if col in df_gt.columns:
        df_gt[col] = pd.to_numeric(df_gt[col], errors='coerce').astype('Int64')
for col in bool_cols_gt:
    if col in df_gt.columns:
        df_gt[col] = df_gt[col].astype(str).str.lower().map({'true': True, 'false': False})

save_metrics(df_gt, [c for c in int_cols_gt if c in df_gt.columns], METRICS_DIR_GT)
target_gt = int(df_gt['is_high_priority_test_gene'].sum()) if 'is_high_priority_test_gene' in df_gt.columns else 0
print(f"genetic_test_ml_features - Target is_high_priority_test_gene : {target_gt:,} ({target_gt/len(df_gt)*100:.1f}%)")
print(f"Saved metrics to {METRICS_DIR_GT}")

---
## 7. protein_family_ml_features and transcript_expression_ml_features
**Supporting tables for protein family and expression use cases**

In [ ]:
IMAGES_DIR_PF, REPORTS_DIR_PF, METRICS_DIR_PF = make_dirs('protein_family_ml_features')
IMAGES_DIR_TE, REPORTS_DIR_TE, METRICS_DIR_TE = make_dirs('transcript_expression_ml_features')

# protein_family_ml_features
print("Loading protein_family_ml_features (full table)...")
df_pfl = pd.read_sql("SELECT * FROM gold.protein_family_ml_features", engine)
print(f"Rows: {len(df_pfl):,}  Cols: {len(df_pfl.columns)}")

int_cols_pfl = [
    'protein_count', 'max_domain_count', 'proteins_with_kinase', 'proteins_with_receptor',
    'proteins_with_zinc_finger', 'proteins_with_sh2', 'proteins_with_sh3',
    'proteins_with_ph', 'proteins_with_death', 'proteins_with_leucine_zipper',
    'proteins_with_helix_loop', 'proteins_with_ig', 'proteins_with_functional_domain',
    'domain_diversity_score', 'functional_complexity_score', 'druggability_potential_score'
]
double_cols_pfl = ['gene_druggability_score']
bool_cols_pfl = [
    'is_kinase', 'is_receptor', 'is_enzyme', 'has_signaling_domain',
    'has_dna_binding_domain', 'has_membrane_domain', 'has_apoptosis_domain',
    'has_immune_domain', 'is_multi_domain_protein', 'is_high_value_protein_family'
]
for col in int_cols_pfl:
    if col in df_pfl.columns:
        df_pfl[col] = pd.to_numeric(df_pfl[col], errors='coerce').astype('Int64')
for col in double_cols_pfl:
    if col in df_pfl.columns:
        df_pfl[col] = pd.to_numeric(df_pfl[col], errors='coerce')
for col in bool_cols_pfl:
    if col in df_pfl.columns:
        df_pfl[col] = df_pfl[col].astype(str).str.lower().map({'true': True, 'false': False})

total_pfl = len(df_pfl)
target_pfl = int(df_pfl['is_high_value_protein_family'].sum()) if 'is_high_value_protein_family' in df_pfl.columns else 0
print(f"Target is_high_value_protein_family : {target_pfl:,} ({target_pfl/total_pfl*100:.1f}%)")
save_metrics(df_pfl, [c for c in int_cols_pfl if c in df_pfl.columns], METRICS_DIR_PF)
print(f"Saved metrics to {METRICS_DIR_PF}")

# transcript_expression_ml_features
print("\nLoading transcript_expression_ml_features (full table)...")
df_te = pd.read_sql("SELECT * FROM gold.transcript_expression_ml_features", engine)
print(f"Rows: {len(df_te):,}  Cols: {len(df_te.columns)}")

int_cols_te = ['gene_length','total_tissues_expressed','tissue_type_count',
               'primary_tissue_count','expression_significance_score','clinical_relevance_score']
double_cols_te = ['max_expression_tpm','avg_expression_tpm','peak_expression_tpm','tissue_specificity_score']
bool_cols_te = ['is_kinase','is_receptor','is_enzyme','is_transcription_factor',
                'is_ubiquitously_expressed','is_tissue_specific','is_highly_expressed',
                'is_lowly_expressed','is_clinically_relevant_expression']
for col in int_cols_te:
    if col in df_te.columns:
        df_te[col] = pd.to_numeric(df_te[col], errors='coerce').astype('Int64')
for col in double_cols_te:
    if col in df_te.columns:
        df_te[col] = pd.to_numeric(df_te[col], errors='coerce')
for col in bool_cols_te:
    if col in df_te.columns:
        df_te[col] = df_te[col].astype(str).str.lower().map({'true': True, 'false': False})

total_te = len(df_te)
target_te = int(df_te['is_clinically_relevant_expression'].sum()) if 'is_clinically_relevant_expression' in df_te.columns else 0
print(f"Target is_clinically_relevant_expression : {target_te:,} ({target_te/total_te*100:.1f}%)")

expr_feats_te = ['max_expression_tpm','avg_expression_tpm','tissue_specificity_score',
                  'total_tissues_expressed','clinical_relevance_score']
expr_feats_te = [c for c in expr_feats_te if c in df_te.columns]
n_cols = 3
n_rows = (len(expr_feats_te) + 2) // 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()
for i, col in enumerate(expr_feats_te):
    data = df_te[col].dropna()
    axes[i].hist(data, bins=40, color='darkorange', alpha=0.8, edgecolor='black')
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Value', fontsize=9)
    axes[i].set_ylabel('Frequency', fontsize=9)
    axes[i].grid(alpha=0.3)
for i in range(len(expr_feats_te), len(axes)):
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(IMAGES_DIR_TE / '01_expression_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: transcript_expression_ml_features/01_expression_score_distributions.png")

save_metrics(df_te, expr_feats_te, METRICS_DIR_TE)
print(f"Saved metrics to {METRICS_DIR_TE}")

## 8. Summary

In [ ]:
print("=" * 70)
print("GENE-LEVEL TABLES EDA COMPLETE")
print("=" * 70)
print()
print("Tables analyzed:")
print(f"  gene_expression_ml_features            : {total_ge:,} rows")
print(f"  gene_pharmacogene_ml_features          : {total_gpg:,} rows  (USE CROSS-VALIDATION)")
print(f"  gene_protein_family_ml_features        : {total_gpf:,} rows")
print(f"  gene_test_availability_ml_features     : {total_gta:,} rows")
print(f"  genetic_test_ml_features               : {len(df_gt):,} rows")
print(f"  protein_family_ml_features             : {total_pfl:,} rows")
print(f"  transcript_expression_ml_features      : {total_te:,} rows")
print()
print("Target variable summary:")
print(f"  UC10 is_high_priority_pharmacogene      : {target_gpg/total_gpg*100:.1f}% positive  (CV needed)")
print(f"  UC11 is_clinically_relevant_expression  : {target_ge/total_ge*100:.1f}% positive")
print(f"  UC12 is_high_value_protein_family       : {target_gpf/total_gpf*100:.1f}% positive")
print(f"  UC13 is_high_priority_test_gene         : {target_gta/total_gta*100:.1f}% positive")
print(f"  UC14 is_clinically_relevant_expression  : {target_te/total_te*100:.1f}% positive  (transcript)")
print()
print("Phase A EDA complete. Next: 09_feature_selection_all_usecases.ipynb")